# Notebook 03 — Idea → Expression Demo (No WQB)

**Goal**: Demonstrate the full idea-generation → synthesis → validation pipeline
**without** calling the WQB simulation API. This lets you iterate on the LLM
prompt quality cheaply.

Pipeline:
```
Research direction
  → IdeaAgent (RAG + LLM) → N hypotheses
  → ExprSynthAgent (LLM) → K×N FASTEXPR expressions
  → ExprValidator → filter invalid
  → NoveltyScorer → novelty scores
  → Display results
```

**Requires**: Notebooks 01 & 02 completed

In [10]:
import asyncio
import os
from pathlib import Path

import nest_asyncio
nest_asyncio.apply()

import litellm
litellm.drop_params = True  # provider-specific unsupported params are silently dropped

from alpha_agent.config import settings
from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.knowledge.alpha_memory import AlphaMemory
from alpha_agent.data.operator_kb import OperatorKB

store = VectorStore()

# If the main DuckDB file is locked by another notebook/kernel,
# fall back to a per-process DB file for this demo notebook.
try:
    memory = AlphaMemory()
except Exception as e:
    fallback_db = settings.duckdb_path.parent / f"alpha_memory_nb03_{os.getpid()}.db"
    print(f"[Notebook03] AlphaMemory lock detected: {e}")
    print(f"[Notebook03] Falling back to: {fallback_db}")
    memory = AlphaMemory(db_path=Path(fallback_db))

kb = OperatorKB()

print("Loaded: store, memory, operator KB")
print(f"  Alpha memory path: {getattr(memory, '_path', 'unknown')}")
print(f"  Alpha memory: {memory.stats()}")

[Notebook03] AlphaMemory lock detected: IO Error: Could not set lock on file "/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/notebooks/data/alpha_memory.db": Conflicting lock is held in /usr/local/Cellar/python@3.11/3.11.11/Frameworks/Python.framework/Versions/3.11/Resources/Python.app/Contents/MacOS/Python (PID 16552) by user weiyanguang. See also https://duckdb.org/docs/stable/connect/concurrency
[Notebook03] Falling back to: data/alpha_memory_nb03_16582.db
Loaded: store, memory, operator KB
  Alpha memory path: data/alpha_memory_nb03_16582.db
  Alpha memory: {'total': 0, 'qualified': 0, 'pass_rate': 0.0}


## 1. Configure the research direction

In [11]:
# ← Change these to explore different directions
DIRECTION = "cross-sectional momentum with volume-adjusted signal strength"
DATASET   = "pv1"
N_IDEAS   = 5
K_VARIANTS = 3

## 2. Generate factor hypotheses via IdeaAgent

In [12]:
import importlib

from alpha_agent.engine import idea_agent as idea_agent_module
importlib.reload(idea_agent_module)
IdeaAgent = idea_agent_module.IdeaAgent

idea_agent = IdeaAgent(store, memory)

ideas = asyncio.run(
    idea_agent.generate_ideas(
        direction=DIRECTION,
        dataset=DATASET,
        n=N_IDEAS,
        explore_exploit_bias=0.3,  # lean toward exploration
    )
)

print(f"Generated {len(ideas)} hypotheses:\n")
for i, idea in enumerate(ideas, 1):
    print(f"[{i}] {idea.get('hypothesis', '')}")
    print(f"    Fields: {idea.get('candidate_fields', [])}")
    print(f"    Ops:    {idea.get('candidate_operators', [])}")
    print(f"    Risk:   {idea.get('risk', '')}")
    print(f"    Novel:  {idea.get('novelty_angle', '')}")
    print()

Generated 5 hypotheses:

[1] Stocks with high past returns and positive rolling correlation between volume and returns exhibit stronger cross-sectional momentum continuation.
    Fields: ['vwap', 'volume']
    Ops:    ['ts_corr', 'ts_sum', 'rank']
    Risk:   High turnover due to rapid changes in volume-return correlations, especially in volatile markets.
    Novel:  Dynamic adjustment of momentum signals using time-varying volume-return correlation, rather than static volume filters.

[2] Pure price momentum isolated by orthogonalizing past returns derived from VWAP with respect to volume trends, then ranked cross-sectionally.
    Fields: ['vwap', 'volume']
    Ops:    ['vector_neut', 'rank']
    Risk:   Data sparsity for volume series in small-cap stocks, leading to unstable orthogonalization and signal degradation.
    Novel:  Using orthogonalization to decouple momentum from volume effects, focusing on residual price moves that may be more predictive.

[3] Cross-sectional momentum 

## 3. Synthesize FASTEXPR expressions

In [13]:
from alpha_agent.engine.expr_synth_agent import ExprSynthAgent

# Use field IDs from vector store as known fields
from alpha_agent.knowledge.vector_store import COLLECTION_FIELDS
field_docs = store._col(COLLECTION_FIELDS).get(include=['metadatas'])
known_fields = {m['field_id'] for m in field_docs['metadatas'] if m.get('field_id')}
known_fields.update({'close', 'open', 'high', 'low', 'volume', 'returns', 'vwap',
                     'turnover', 'sharesout', 'cap'})
print(f"Known field count: {len(known_fields)}")

synth = ExprSynthAgent(kb, memory, known_fields=known_fields)
all_fields_list = list(known_fields)

all_expressions = {}

for idea in ideas:
    exprs = asyncio.run(
        synth.synthesize(idea, all_fields=all_fields_list, k=K_VARIANTS)
    )
    all_expressions[idea['id']] = exprs
    print(f"[{idea['id']}] {len(exprs)} expressions generated")
    for e in exprs:
        print(f"   {e}")
    print()

Known field count: 170
[volume_corr_momentum] 3 expressions generated
   rank(ts_sum(returns, 20)) * if_else(ts_corr(volume, returns, 10) > 0, ts_corr(volume, returns, 10), 0)
   scale(ts_sum(returns, 15) * ts_corr(volume, returns, 20) * if_else(ts_corr(volume, returns, 20) > 0, 1, 0))
   rank(ts_sum(returns, 30)) * rank(if_else(ts_corr(volume, returns, 15) > 0, ts_corr(volume, returns, 15), 0))

[ExprSynthAgent] LLM call failed: Expecting value: line 1 column 1 (char 0)
[ExprSynthAgent] LLM call failed: Expecting value: line 1 column 1 (char 0)
[ortho_volume_momentum] 3 expressions generated
   rank(vector_neut((vwap/delay(vwap,20)-1), ts_mean(volume,20)))
   rank(vector_neut(ts_rank(vwap,60), ts_sum(volume,10)))
   rank(vector_neut((vwap-delay(vwap,5))/ts_std_dev(vwap,10), delta(volume,5)))

[volume_rank_weighted_momentum] 3 expressions generated
   ts_sum(returns, 20) * rank(ts_mean(volume, 20))
   (ts_sum(returns, 10) / (ts_std_dev(returns, 10) + 1e-6)) * rank(ts_sum(volume, 10) * 

## 4. Validate expressions locally

In [14]:
from alpha_agent.engine.validator import ExprValidator

validator = ExprValidator(kb, known_fields=known_fields)

flat_exprs = [e for exprs in all_expressions.values() for e in exprs]

valid_exprs = []
invalid_exprs = []

print(f"Validating {len(flat_exprs)} expressions...\n")
for expr in flat_exprs:
    result = validator.validate(expr)
    status = "✓ PASS" if result.ok else "✗ FAIL"
    print(f"{status}: {expr[:80]}")
    if result.errors:
        for e in result.errors:
            print(f"         ERROR: {e}")
    if result.warnings:
        for w in result.warnings:
            print(f"         WARN:  {w}")
    if result.ok:
        valid_exprs.append(expr)
    else:
        invalid_exprs.append(expr)

print(f"\nSummary: {len(valid_exprs)} valid, {len(invalid_exprs)} invalid")

Validating 15 expressions...

✓ PASS: rank(ts_sum(returns, 20)) * if_else(ts_corr(volume, returns, 10) > 0, ts_corr(vo
✓ PASS: scale(ts_sum(returns, 15) * ts_corr(volume, returns, 20) * if_else(ts_corr(volum
✓ PASS: rank(ts_sum(returns, 30)) * rank(if_else(ts_corr(volume, returns, 15) > 0, ts_co
✓ PASS: rank(vector_neut((vwap/delay(vwap,20)-1), ts_mean(volume,20)))
✓ PASS: rank(vector_neut(ts_rank(vwap,60), ts_sum(volume,10)))
✓ PASS: rank(vector_neut((vwap-delay(vwap,5))/ts_std_dev(vwap,10), delta(volume,5)))
✓ PASS: ts_sum(returns, 20) * rank(ts_mean(volume, 20))
✓ PASS: (ts_sum(returns, 10) / (ts_std_dev(returns, 10) + 1e-6)) * rank(ts_sum(volume, 1
✓ PASS: ts_rank(close, 60) * rank(volume / (ts_mean(volume, 60) + 1e-6))
✓ PASS: rank(ts_sum(if_else(volume > ts_mean(volume, 20), returns, 0), 10))
✓ PASS: rank(ts_sum(if_else(volume > ts_std_dev(volume, 30) + ts_mean(volume, 30), retur
✓ PASS: rank(ts_mean(if_else(volume > ts_mean(volume, 25), returns, 0), 12))
✓ PASS: rank(ts_corr(vol

## 5. Novelty scoring

In [15]:
from alpha_agent.search.novelty import NoveltyScorer

novelty = NoveltyScorer(store, memory)
scores = novelty.score_batch(valid_exprs)

print("Novelty scores (higher = more novel):\n")
for expr, score in sorted(zip(valid_exprs, scores), key=lambda x: -x[1]):
    bar = '█' * int(score * 20)
    print(f"  {score:.3f} {bar}")
    print(f"         {expr[:90]}")
    print()

Novelty scores (higher = more novel):

  1.000 ████████████████████
         rank(ts_sum(returns, 20)) * if_else(ts_corr(volume, returns, 10) > 0, ts_corr(volume, retu

  1.000 ████████████████████
         scale(ts_sum(returns, 15) * ts_corr(volume, returns, 20) * if_else(ts_corr(volume, returns

  1.000 ████████████████████
         rank(ts_sum(returns, 30)) * rank(if_else(ts_corr(volume, returns, 15) > 0, ts_corr(volume,

  1.000 ████████████████████
         rank(vector_neut((vwap/delay(vwap,20)-1), ts_mean(volume,20)))

  1.000 ████████████████████
         rank(vector_neut(ts_rank(vwap,60), ts_sum(volume,10)))

  1.000 ████████████████████
         rank(vector_neut((vwap-delay(vwap,5))/ts_std_dev(vwap,10), delta(volume,5)))

  1.000 ████████████████████
         ts_sum(returns, 20) * rank(ts_mean(volume, 20))

  1.000 ████████████████████
         (ts_sum(returns, 10) / (ts_std_dev(returns, 10) + 1e-6)) * rank(ts_sum(volume, 10) * ts_me

  1.000 ████████████████████
         ts_r

In [16]:
# Filter by novelty threshold
from alpha_agent.config import settings

novel_exprs = novelty.filter_novel(valid_exprs, threshold=settings.novelty_score_min)
print(f"After novelty filter (threshold={settings.novelty_score_min}):")
print(f"  {len(novel_exprs)} / {len(valid_exprs)} expressions pass\n")
for e in novel_exprs:
    print(f"  → {e}")

After novelty filter (threshold=0.3):
  15 / 15 expressions pass

  → rank(ts_sum(returns, 20)) * if_else(ts_corr(volume, returns, 10) > 0, ts_corr(volume, returns, 10), 0)
  → scale(ts_sum(returns, 15) * ts_corr(volume, returns, 20) * if_else(ts_corr(volume, returns, 20) > 0, 1, 0))
  → rank(ts_sum(returns, 30)) * rank(if_else(ts_corr(volume, returns, 15) > 0, ts_corr(volume, returns, 15), 0))
  → rank(vector_neut((vwap/delay(vwap,20)-1), ts_mean(volume,20)))
  → rank(vector_neut(ts_rank(vwap,60), ts_sum(volume,10)))
  → rank(vector_neut((vwap-delay(vwap,5))/ts_std_dev(vwap,10), delta(volume,5)))
  → ts_sum(returns, 20) * rank(ts_mean(volume, 20))
  → (ts_sum(returns, 10) / (ts_std_dev(returns, 10) + 1e-6)) * rank(ts_sum(volume, 10) * ts_mean(close, 10))
  → ts_rank(close, 60) * rank(volume / (ts_mean(volume, 60) + 1e-6))
  → rank(ts_sum(if_else(volume > ts_mean(volume, 20), returns, 0), 10))
  → rank(ts_sum(if_else(volume > ts_std_dev(volume, 30) + ts_mean(volume, 30), returns, 0), 1

## ✅ Notebook 03 Complete

You've seen the full idea → expression → validate → novelty pipeline.
No WQB API calls were made — all results are from the LLM + local checks.

The novel expressions above are ready for WQB simulation in Notebook 04.